# TraceWin distribution viewer

This notebook reads TraceWin binary `.dst` distributions and plots the transverse phase spaces.

It uses the configured TraceWin workspace and automatically selects the first valid calc directory containing `partran1.out` and `.dst` files.
Run all cells after each TraceWin/PARTRAN run so the files are reloaded from disk.

In [ ]:
# Basic imports.
# pathlib handles file paths cleanly; numpy/pandas/matplotlib are used for data and plots.
import os
import re
import sys
from datetime import datetime
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/beam_optimization_matplotlib')

import numpy as np
import matplotlib.pyplot as plt

# Make the beam_optimization package importable when this notebook is run directly.
def find_project_root(start_path=None):
    start = Path.cwd() if start_path is None else Path(start_path).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'beam_optimization' / '__init__.py').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find the repository root containing beam_optimization/__init__.py. '
        'Start Jupyter from the rl_beam_optimization repository or set PROJECT_ROOT manually.'
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def safe_filename(text):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(text)).strip('_')


# Change this path if you want the PNG files in another folder.
IMAGE_OUTPUT_DIR = PROJECT_ROOT / 'beam_optimization' / 'output'
IMAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from beam_optimization.env.tracewin_env.tracewin.visualization import (
    find_final_tracewin_dst_path,
    plot_tracewin_distribution,
    tracewin_distribution_from_dst,
)
from beam_optimization.config.paths import DEFAULT_TRACEWIN_INI

# TraceWin run paths.
# The new project check writes to /tmp/tracewin_check. Older/debug notebooks may write
# to TraceWin_workspace/calc_debug_env or TraceWin_workspace/calc. Pick the first one
# that has both partran1.out and at least one .dst file.
TRACEWIN_WORKSPACE = Path(DEFAULT_TRACEWIN_INI).parent
CALC_CANDIDATES = [
    Path('/tmp/tracewin_check'),
    TRACEWIN_WORKSPACE / 'calc_debug_env',
    TRACEWIN_WORKSPACE / 'calc',
]


def calc_dir_is_valid(path):
    return (path / 'partran1.out').exists() and any(path.glob('*.dst'))


CALC_DIR = next((path for path in CALC_CANDIDATES if calc_dir_is_valid(path)), None)
if CALC_DIR is None:
    searched = '\n'.join(f'  - {path}' for path in CALC_CANDIDATES)
    message = (
        'No TraceWin calc directory with partran1.out and .dst files found.\n'
        'Run `python -m beam_optimization check` to generate /tmp/tracewin_check, '
        'or run a TraceWin test/debug notebook first, then rerun this cell.\n'
        f'Searched:\n{searched}'
    )
    raise FileNotFoundError(message)

DST_INPUT = TRACEWIN_WORKSPACE / '16O5.dst'
DST_OUTPUT = find_final_tracewin_dst_path(CALC_DIR)
PARTRAN_FILE = CALC_DIR / 'partran1.out'

# Print the exact files used, so it is obvious which TraceWin run is being plotted.
print(f'CALC_DIR    = {CALC_DIR}')
print(f'DST_INPUT   = {DST_INPUT}')
print(f'DST_OUTPUT  = {DST_OUTPUT}')
print(f'PARTRAN_OUT = {PARTRAN_FILE}')
print(f'PNG_OUTPUT  = {IMAGE_OUTPUT_DIR}')


In [ ]:
def natural_key(path):
    # Sort filenames naturally: 2.dst comes before 11.dst.
    parts = re.split(r'(\d+)', Path(path).name)
    return [int(part) if part.isdigit() else part.lower() for part in parts]


def read_npart_simulated(partran_file):
    # Read the number of simulated macro-particles from the partran1.out header.
    with open(partran_file) as f:
        f.readline()
        values = f.readline().split()
    return int(values[-1])


NPART_SIMULATED = read_npart_simulated(PARTRAN_FILE)

# Intermediate distributions are every .dst file written in calc/ by PLOT_DST,
# excluding TraceWin's standard input/output names and the selected final output.
excluded_names = {'part_rfq.dst', 'part_dtl1.dst'}
intermediate_files = [
    path for path in sorted(CALC_DIR.glob('*.dst'), key=natural_key)
    if path.name not in excluded_names and path != DST_OUTPUT
]

# Plot order: reservoir input, PLOT_DST snapshots, final output if available.
distribution_files = [DST_INPUT, *intermediate_files]
if DST_OUTPUT is not None:
    distribution_files.append(DST_OUTPUT)

missing_files = [path for path in distribution_files if not path.exists()]
if missing_files:
    print('Skipped missing distribution files:')
    for path in missing_files:
        print(f'  {path}')
distribution_files = [path for path in distribution_files if path.exists()]

distributions = []
for path in distribution_files:
    role = 'input' if path == DST_INPUT else 'output' if path == DST_OUTPUT else 'intermediate'

    # Figure title: explicit names for the endpoints; raw filename for intermediate PLOT_DST files.
    display_title = {
        'input': 'Input distribution',
        'output': 'Output distribution',
        'intermediate': path.name,
    }[role]

    distribution = tracewin_distribution_from_dst(path)
    distributions.append({
        'title': display_title,
        'filename': path.name,
        'role': role,
        'path': path,
        'distribution': distribution,
        'n_particles': len(distribution['x']),
    })

print(f'NPART_SIMULATED = {NPART_SIMULATED:,}')
print('Loaded distributions:')
for item in distributions:
    modified = datetime.fromtimestamp(item['path'].stat().st_mtime)
    print(f"  {item['title']:<20} {item['filename']:<14} {item['role']:<12} {item['n_particles']:>10,} particles   modified: {modified}")


In [ ]:
# Distribution plotting is centralized in:
# beam_optimization.env.tracewin_env.tracewin.visualization.plot_tracewin_distribution
# The same function is used by TraceWinEnv.render_final_beam_distribution().


In [ ]:
# partran1.out contains element-by-element beam statistics.
# We use it here for z positions and for the aperture column in the x-y plots.
def read_partran_table(partran_file):
    with open(partran_file) as f:
        lines = f.readlines()

    header_line_index = next(
        i for i, line in enumerate(lines)
        if line.lstrip().startswith('##') and 'z(m)' in line
    )
    columns = lines[header_line_index].replace('##', '', 1).split()
    data = np.loadtxt(partran_file, skiprows=header_line_index + 1)

    # TraceWin writes an unnamed first column with the element index before z(m).
    # If we do not account for it, SizeX/SizeY/Aper are shifted by one column.
    if data.ndim == 1:
        data = data.reshape(1, -1)
    if data.shape[1] == len(columns) + 1:
        columns = ['element', *columns]
    elif data.shape[1] != len(columns):
        raise ValueError(
            f'Unexpected partran table shape: {data.shape[1]} values, '
            f'{len(columns)} header columns'
        )
    return {name: data[:, i] for i, name in enumerate(columns)}


partran = read_partran_table(PARTRAN_FILE)
z = partran['z(m)']
TUBE_RADIUS_MM = float(np.nanmax(partran['Aper']))
PLOT_XY_RANGE_MM = TUBE_RADIUS_MM  # Use 50.0 here for a zoomed beam view.
PLOT_ANGLE_RANGE_MRAD = float(np.ceil(max(
    np.nanmax(np.abs(item['distribution'][key] * 1000.0))
    for item in distributions
    for key in ('xp', 'yp')
) / 10.0) * 10.0)

print(f'Loaded {len(z)} rows from {PARTRAN_FILE.name}')
print(f'TUBE_RADIUS_MM = {TUBE_RADIUS_MM:g}')
print(f'PLOT_XY_RANGE_MM = {PLOT_XY_RANGE_MM:g}')
print(f'PLOT_ANGLE_RANGE_MRAD = {PLOT_ANGLE_RANGE_MRAD:g}')

In [ ]:
# Draw one figure for each distribution.
# Input/output use readable endpoint titles; intermediate snapshots use their calc filename.
saved_distribution_figures = []

for i, item in enumerate(distributions):
    if item['role'] == 'input':
        subtitle = f"{item['n_particles']:,} particles in reservoir, {NPART_SIMULATED:,} simulated"
    elif item['role'] == 'output':
        subtitle = f"{item['n_particles']:,} particles"
    else:
        subtitle = f"{item['n_particles']:,} particles"

    save_path = IMAGE_OUTPUT_DIR / f"distribution_{i:02d}_{safe_filename(item['filename'])}.png"
    plot_tracewin_distribution(
        item['distribution'],
        title=f"{item['title']} - {subtitle}",
        figure_name=f"TraceWin distribution - {item['filename']}",
        bins=200,
        xy_range_mm=PLOT_XY_RANGE_MM,
        angle_range_mrad=PLOT_ANGLE_RANGE_MRAD,
        aperture_radius_mm=TUBE_RADIUS_MM,
        figsize=(22, 6),
        save_path=save_path,
        show=True,
    )
    saved_distribution_figures.append(save_path)

print('Saved distribution figures:')
for path in saved_distribution_figures:
    print(f'  {path}')


## Envelope plot

In [ ]:
# Envelope plot from partran1.out.
# Black curves are aperture walls; colored bands show centroid +/- RMS size.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, plane, centroid_col, size_col, color in [
    (axes[0], 'X', 'x0', 'SizeX', 'steelblue'),
    (axes[1], 'Y', 'y0', 'SizeY', 'tomato'),
]:
    c0 = partran[centroid_col]
    size = partran[size_col]
    aper = partran['Aper']

    ax.plot(z, aper, color='black', linewidth=2.0, label='aperture')
    ax.plot(z, -aper, color='black', linewidth=2.0)

    ax.fill_between(z, c0 - size, c0 + size, alpha=0.35, color=color)
    ax.plot(z, c0 + size, color=color, linewidth=1.5, label='RMS envelope (+/-1 sigma)')
    ax.plot(z, c0 - size, color=color, linewidth=1.5)
    ax.plot(z, c0, color='black', linewidth=0.8, linestyle=':', label='centroid')

    ax.axhline(0, color='lightgray', lw=0.5)
    ax.set_xlabel('Position [m]', fontsize=10)
    ax.set_ylabel(f'{plane} [mm]', fontsize=10)
    ax.set_title(f'{plane} plane', fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.suptitle('Beam RMS envelope along the line', fontsize=12)
plt.tight_layout()
envelope_path = IMAGE_OUTPUT_DIR / 'beam_rms_envelope.png'
fig.savefig(envelope_path, dpi=160)
print(f'Saved envelope figure: {envelope_path}')
plt.show()